Step 1 — Install & Load Dataset

Step 2 — Tokenization + Binary Storage

In [ ]:
!pip install datasets tiktoken

from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

In [ ]:
import tiktoken
import numpy as np
import os
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example["text"])
    return {"ids": ids, "len": len(ids)}

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=["text"],
        desc="Tokenizing dataset",
        num_proc=8,
    )

    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = f"{split}.bin"
        dtype = np.uint16

        arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
        idx = 0

        for sample in tqdm(dset):
            ids = np.array(sample["ids"], dtype=dtype)
            arr[idx:idx+len(ids)] = ids
            idx += len(ids)

        arr.flush()

Step 3 — Create TensorFlow Dataset

In [ ]:
import tensorflow as tf

block_size = 128
batch_size = 32

def get_dataset(split):
    filename = "train.bin" if split == "train" else "validation.bin"
    data = np.memmap(filename, dtype=np.uint16, mode="r")

    def generator():
        while True:
            ix = np.random.randint(0, len(data) - block_size)
            x = np.array(data[ix:ix+block_size], dtype=np.int32)
            y = np.array(data[ix+1:ix+1+block_size], dtype=np.int32)
            yield x, y

    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(block_size,), dtype=tf.int32),
            tf.TensorSpec(shape=(block_size,), dtype=tf.int32),
        )
    ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = get_dataset("train")
val_ds = get_dataset("validation")

Step 4 — Define GPT Model in Keras

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    mask = tf.cast(i >= j - n_src + n_dest, dtype="int32")
    mask = tf.reshape(mask, (1, n_dest, n_src))
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1),
         tf.constant([1, 1], dtype=tf.int32)], axis=0)
    return tf.tile(mask, mult)

Transformer Block

In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(4 * embed_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, inputs, training=False):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]

        causal_mask = causal_attention_mask(batch_size, seq_len, seq_len)

        attn_output = self.att(
            inputs,
            inputs,
            attention_mask=causal_mask,
            training=training
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        return self.layernorm2(out1 + ffn_output)

GPT Model

In [ ]:
import tensorflow as tf

class GPT(tf.keras.Model):
    def __init__(self, vocab_size, block_size,
                 n_layer=6, n_head=6, n_embd=384, dropout=0.1):
        super().__init__()

        self.block_size = block_size
        self.vocab_size = vocab_size
        self.n_embd = n_embd

        # Token embedding
        self.token_emb = tf.keras.layers.Embedding(vocab_size, n_embd)

        # Positional embedding
        self.pos_emb = tf.keras.layers.Embedding(block_size, n_embd)

        self.drop = tf.keras.layers.Dropout(dropout)

        self.blocks = [
            TransformerBlock(n_embd, n_head, dropout)
            for _ in range(n_layer)
        ]

        self.ln_f = tf.keras.layers.LayerNormalization(epsilon=1e-5)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        positions = tf.range(0, seq_len)

        tok_emb = self.token_emb(x)
        pos_emb = self.pos_emb(positions)

        x = tok_emb + pos_emb
        x = self.drop(x, training=training)

        for block in self.blocks:
            x = block(x, training=training)

        x = self.ln_f(x)

        # 🔥 WEIGHT TYING HERE
        logits = tf.matmul(x, self.token_emb.embeddings, transpose_b=True)

        return logits

Cosine Decay + Linear Warmup

In [ ]:
class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, min_lr, warmup_steps, total_steps):
        super().__init__()
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        # Linear warmup
        warmup_lr = self.max_lr * (step / self.warmup_steps)

        # Cosine decay
        progress = (step - self.warmup_steps) / (
            self.total_steps - self.warmup_steps
        )
        cosine_decay = 0.5 * (1 + tf.cos(tf.constant(3.1415926535) * progress))
        decay_lr = self.min_lr + (self.max_lr - self.min_lr) * cosine_decay

        return tf.where(step < self.warmup_steps, warmup_lr, decay_lr)

Step 5 — Compile Model

In [ ]:
vocab_size = 50257
epochs = 10
steps_per_epoch = 500
total_steps = epochs * steps_per_epoch
batch_size = 32
weight_decay = 0.1

model = GPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1
)

lr_schedule = WarmupCosineSchedule(
    max_lr=learning_rate,
    min_lr=min_lr,
    warmup_steps=warmup_steps,
    total_steps=total_steps
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-9
)

model = GPT(vocab_size, block_size)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)


Step 6 — Train

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model.weights.keras",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch,
    validation_steps=100,
    callbacks=callbacks
)

)

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation loss
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plot learning rate over epochs
plt.figure(figsize=(12, 6))
plt.plot(history.history['lr'], label='Learning Rate')
plt.title('Learning Rate Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True)
plt.show()


Step 7 — Text Generation

In [ ]:
import numpy as np

def generate(model, prompt, max_tokens=200, temperature=1.0):
    input_ids = enc.encode_ordinary(prompt)
    input_ids = tf.expand_dims(input_ids, 0)

    for _ in range(max_tokens):
        logits = model(input_ids)
        logits = logits[:, -1, :] / temperature
        probs = tf.nn.softmax(logits)
        next_id = tf.random.categorical(tf.math.log(probs), num_samples=1)
        input_ids = tf.concat([input_ids, next_id], axis=1)

        if tf.shape(input_ids)[1] > block_size:
            input_ids = input_ids[:, -block_size:]

    return enc.decode(input_ids.numpy().squeeze())

print(generate(model, "Once upon a time there was a pumpkin."))